# Римановский случайный лес

Каждое «дерево» — маленькая римановская модель (перецентрирование + касательное пространство + LogReg), которая видит:
- случайную частотную полосу из банка,
- случайные `n_channels` каналов: подматрица ковариации снова SPD, так что риманов пайплайн работает как есть,
- бутстреп *участников*: учит устойчивости к смене человека.

Лес усредняет вероятности деревьев: $p(y\mid X) = \frac1K\sum_k p_k(y\mid X)$.
Код: `bci/models/forest.py`.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from bci.data import load_split, concat
from bci.evaluation import score, evaluate, cross_validation_summary, compare_with_baselines
from bci.models import forest

train, val, test = (load_split(s, "../data") for s in ["train", "val", "test"])
dev = concat(train, val)  # для кросс-валидации; test только в самом конце

PARAMS = dict(n_trees=50, n_channels=20, C=0.01, bands=forest.BANDS, prior_weight=5)
fit = lambda tr: forest.fit(tr, **PARAMS)

## Сила толпы
Слева: одно дерево слабое (гистограмма), а лес из первых $k$ деревьев растёт с $k$. Справа: средний скор одного дерева по полосам.

In [ ]:
model = fit(train)
P = model.tree_probas(val)  # (n_trees, N, 3)
single = [score(val, p.argmax(-1))["session_mean"] for p in P]
growing = [score(val, P[:k].mean(0).argmax(-1))["session_mean"] for k in range(1, len(P) + 1)]
bands = [f"{lo}-{hi} Hz" for lo, hi in PARAMS["bands"]]

fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.5))
a.plot(range(1, len(P) + 1), growing, label="лес из первых k деревьев")
a.hist(single, bins=15, orientation="horizontal", alpha=0.4, weights=np.full(len(single), 2), label="одно дерево")
a.axhline(1 / 3, ls="--", c="gray"); a.set(xlabel="k", title="val session_mean"); a.legend()
by_band = pd.Series(single).groupby([bands[t.band] for t in model.trees]).mean()
by_band.plot.barh(ax=b, title="одно дерево по полосам (val)"); b.axvline(1 / 3, ls="--", c="gray")
plt.tight_layout()

## Кросс-валидация на train + val
По ней сравниваем гипотезы: новые люди (каждый по очереди в тесте) и поздние сессии знакомых людей.

In [ ]:
cross_validation_summary(fit, dev)

## Официальный test
Обучение на train, одна оценка на test, рядом опубликованные бейзлайны. `evaluate` повторяет их оценку в точности.

In [ ]:
compare_with_baselines({"Riemannian forest": evaluate(fit(train), test)})